
# SESSÃO 12 — AUDITORIA DE CLASSES E PREPARAÇÃO DO TREINAMENTO V4

**Projeto:** Classificação supervisionada de eventos indesejados — 3W 2.0.0 (C1).

**Data:** 25/11/2026.

## Objetivo

Estabelecer os critérios metodológicos para o treinamento
supervisionado utilizando a matriz de atributos V4.

## Fundamentação

A Sessão 11 produziu 321.929 janelas temporais distribuídas
em 27 poços, provenientes de 478 arquivos de treinamento.

A distribuição dos rótulos temporais evidenciou desequilíbrio
entre classes, concentração de janelas em determinados poços
e baixa representatividade de alguns eventos desenvolvidos.

Conforme as orientações do professor no Checkpoint 1,
essas limitações devem ser investigadas e documentadas
antes do treinamento definitivo.

## Configuração experimental

- Particionamento: V4 ajustada e compartilhada entre C1 e C2.
- Conjunto principal desta sessão: treinamento.
- Janela temporal: 100 amostras.
- Deslocamento: 50 amostras.
- Mínimo de dois dos cinco sensores completos.
- Preservação dos rótulos temporais originais.
- Validação interna independente por poço.
- Conjunto de teste preservado.

## Resultado esperado

Definição fundamentada das classes elegíveis, do tratamento
dos transientes, da estratégia de validação e do tratamento
dos atributos ausentes, sem vazamento de informações.


In [ ]:

# SESSAO 12 - TAREFA 01
# Recuperacao e conferencia da matriz de atributos V4

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")

raiz = Path("/content/drive/MyDrive")

# Localizar a auditoria da Sessao 10
auditorias = list(
    raiz.rglob("10_Auditoria_535_Arquivos_V4.csv")
)

assert len(auditorias) == 1, auditorias

pasta_exp02 = auditorias[0].parent
pasta_matriz = pasta_exp02 / "11_Matriz_Atributos_V4"

assert pasta_matriz.exists()

# Recuperar o controle da extracao
controle = pd.read_csv(
    pasta_matriz / "11_Progresso_Extracao.csv"
)

aud = pd.read_csv(auditorias[0])

# Conferir os resultados anteriores
assert len(controle) == 535
assert controle["Arquivo"].is_unique
assert controle["Status"].eq("OK").all()
assert controle["Janelas"].sum() == 321929

assert aud["Arquivo"].is_unique
assert set(controle["Arquivo"]) == set(aud["Arquivo"])

esperados = aud.set_index("Arquivo")["Janelas_2_ou_mais"]

assert (
    controle["Arquivo"].map(esperados) ==
    controle["Janelas"]
).all()

# Selecionar arquivos com janelas validas
validos = controle[
    controle["Janelas"] > 0
].copy()

assert len(validos) == 478

# Recuperar a matriz completa
partes = []

for _, item in tqdm(
    validos.iterrows(),
    total=len(validos),
    desc="Carregando matriz V4"
):

    caminho = pasta_matriz / (
        Path(item["Arquivo"]).stem +
        "_atributos.parquet"
    )

    assert caminho.exists(), caminho

    df = pd.read_parquet(caminho)

    assert len(df) == int(item["Janelas"])
    assert df.shape[1] == 37
    assert df["Arquivo"].eq(item["Arquivo"]).all()
    assert df["Poco"].eq(item["Poco"]).all()

    partes.append(df)

matriz_v4 = pd.concat(
    partes,
    ignore_index=True
)

del partes, df

# Configuracao dos atributos
SENSORES = [
    "P-TPT",
    "T-TPT",
    "P-ANULAR",
    "P-JUS-CKGL",
    "P-MON-CKP"
]

ESTATISTICAS = [
    "media", "desvio", "inclinacao", "p10", "p90"
]

atributos = [
    f"{s}_{a}"
    for s in SENSORES
    for a in ESTATISTICAS
]

indicadores = [
    f"{s}_disponivel"
    for s in SENSORES
]

preditores = atributos + indicadores

# Conferencia final
assert matriz_v4.shape == (321929, 37)
assert matriz_v4["Poco"].nunique() == 27
assert matriz_v4["Arquivo"].nunique() == 478
assert matriz_v4["N_sensores"].ge(2).all()
assert len(preditores) == 30
assert set(preditores).issubset(matriz_v4.columns)

assert matriz_v4["Rotulo_original"].isin(
    list(range(10)) + list(range(101, 110))
).all()

assert not matriz_v4.duplicated(
    subset=["Arquivo", "Inicio", "Fim"]
).any()

print("\n1. MATRIZ V4 RECUPERADA:")
print("Janelas:", len(matriz_v4))
print("Colunas:", matriz_v4.shape[1])
print("Arquivos:", matriz_v4["Arquivo"].nunique())
print("Pocos:", matriz_v4["Poco"].nunique())

print("\n2. ESTRUTURA DOS PREDITORES:")
print("Atributos estatisticos:", len(atributos))
print("Indicadores:", len(indicadores))
print("Total de preditores:", len(preditores))

print("\n3. CONSUMO DE MEMORIA:")
print(
    round(
        matriz_v4.memory_usage(deep=True).sum() / 1024**2,
        2
    ),
    "MB"
)

print("\n4. DIRETORIO DA MATRIZ:")
print(pasta_matriz)

print("\nTAREFA 01 EXECUTADA")


Mounted at /content/drive


Carregando matriz V4:   0%|          | 0/478 [00:00<?, ?it/s]


1. MATRIZ V4 RECUPERADA:
Janelas: 321929
Colunas: 37
Arquivos: 478
Pocos: 27

2. ESTRUTURA DOS PREDITORES:
Atributos estatisticos: 25
Indicadores: 5
Total de preditores: 30

3. CONSUMO DE MEMORIA:
166.1 MB

4. DIRETORIO DA MATRIZ:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/11_Matriz_Atributos_V4

TAREFA 01 EXECUTADA


In [ ]:

# SESSAO 12 - TAREFA 02
# Auditoria da representatividade dos rotulos temporais

rotulos_possiveis = list(range(10)) + list(range(101, 110))

# 1. Representatividade por rotulo
auditoria_classes = (
    matriz_v4.groupby("Rotulo_original")
    .agg(
        Janelas=("Arquivo", "size"),
        Arquivos=("Arquivo", "nunique"),
        Pocos=("Poco", "nunique")
    )
    .reindex(rotulos_possiveis, fill_value=0)
    .reset_index()
    .rename(columns={"Rotulo_original": "Rotulo"})
)

auditoria_classes["Percentual"] = (
    auditoria_classes["Janelas"] /
    len(matriz_v4) * 100
).round(4)

auditoria_classes["Fase"] = np.select(
    [
        auditoria_classes["Rotulo"] == 0,
        auditoria_classes["Rotulo"] >= 101
    ],
    [
        "Normal",
        "Transiente"
    ],
    default="Evento desenvolvido"
)

# 2. Distribuicao das janelas por rotulo e poco
janelas_por_poco = pd.crosstab(
    matriz_v4["Rotulo_original"],
    matriz_v4["Poco"]
).reindex(rotulos_possiveis, fill_value=0)

# 3. Concentracao das janelas no poco dominante
auditoria_classes["Poco_dominante"] = (
    janelas_por_poco.idxmax(axis=1).values
)

auditoria_classes["Janelas_poco_dominante"] = (
    janelas_por_poco.max(axis=1).values
)

auditoria_classes["Concentracao_max_pct"] = (
    auditoria_classes["Janelas_poco_dominante"] /
    auditoria_classes["Janelas"].replace(0, np.nan)
    * 100
).round(2)

auditoria_classes.loc[
    auditoria_classes["Janelas"] == 0,
    "Poco_dominante"
] = "Ausente"

# 4. Diagnostico preliminar da cobertura
auditoria_classes["Diagnostico"] = np.select(
    [
        auditoria_classes["Pocos"] == 0,
        auditoria_classes["Pocos"] == 1,
        auditoria_classes["Pocos"] == 2
    ],
    [
        "Ausente",
        "Apenas um poco",
        "Apenas dois pocos"
    ],
    default="Tres ou mais pocos"
)

# 5. Identificar os pocos de cada rotulo
pocos_por_rotulo = (
    matriz_v4.groupby("Rotulo_original")["Poco"]
    .apply(lambda x: ", ".join(sorted(x.unique())))
)

auditoria_classes["Lista_pocos"] = (
    auditoria_classes["Rotulo"]
    .map(pocos_por_rotulo)
    .fillna("Ausente")
)

# Conferencias
assert auditoria_classes["Janelas"].sum() == 321929

assert (
    janelas_por_poco.sum(axis=1).values ==
    auditoria_classes["Janelas"].values
).all()

# Apresentar resultados
print("1. REPRESENTATIVIDADE DOS ROTULOS:")
display(
    auditoria_classes[
        [
            "Rotulo", "Fase", "Janelas",
            "Arquivos", "Pocos", "Percentual",
            "Diagnostico"
        ]
    ]
)

print("\n2. CONCENTRACAO POR POCO:")
display(
    auditoria_classes[
        [
            "Rotulo", "Poco_dominante",
            "Janelas_poco_dominante",
            "Concentracao_max_pct"
        ]
    ]
)

print("\n3. POCOS ASSOCIADOS A CADA ROTULO:")
for _, linha in auditoria_classes.iterrows():
    print(
        f"Rotulo {linha['Rotulo']:3d}: "
        f"{linha['Lista_pocos']}"
    )

# 6. Salvar os resultados
pasta_s12 = pasta_exp02 / "12_Auditoria_Classes_V4"
pasta_s12.mkdir(exist_ok=True)

auditoria_classes.to_csv(
    pasta_s12 / "12_Auditoria_Representatividade_Classes.csv",
    index=False
)

janelas_por_poco.to_csv(
    pasta_s12 / "12_Distribuicao_Rotulos_por_Poco.csv"
)

print("\nRESULTADOS SALVOS EM:")
print(pasta_s12)

print("\nTAREFA 02 EXECUTADA")


1. REPRESENTATIVIDADE DOS ROTULOS:


,Rotulo,Fase,Janelas,Arquivos,Pocos,Percentual,Diagnostico
0,0,Normal,138404,304,27,42.9921,Tres ou mais pocos
1,1,Evento desenvolvido,58,1,1,0.0180,Apenas um poco
2,2,Evento desenvolvido,299,7,5,0.0929,Tres ou mais pocos
3,3,Evento desenvolvido,10967,31,1,3.4067,Apenas um poco
4,4,Evento desenvolvido,20211,143,3,6.2781,Tres ou mais pocos
5,5,Evento desenvolvido,68,3,1,0.0211,Apenas um poco
6,6,Evento desenvolvido,248,3,1,0.0770,Apenas um poco
7,7,Evento desenvolvido,34,1,1,0.0106,Apenas um poco
8,8,Evento desenvolvido,1856,7,7,0.5765,Tres ou mais pocos
9,9,Evento desenvolvido,309,2,2,0.0960,Apenas dois pocos



2. CONCENTRACAO POR POCO:


,Rotulo,Poco_dominante,Janelas_poco_dominante,Concentracao_max_pct
0,0,WELL-00002,73274,52.94
1,1,WELL-00002,58,100.00
2,2,WELL-00011,258,86.29
3,3,WELL-00014,10967,100.00
4,4,WELL-00002,15838,78.36
5,5,WELL-00020,68,100.00
6,6,WELL-00002,248,100.00
7,7,WELL-00021,34,100.00
8,8,WELL-00025,755,40.68
9,9,WELL-00042,305,98.71



3. POCOS ASSOCIADOS A CADA ROTULO:
Rotulo   0: WELL-00002, WELL-00007, WELL-00009, WELL-00011, WELL-00012, WELL-00013, WELL-00014, WELL-00020, WELL-00021, WELL-00022, WELL-00023, WELL-00025, WELL-00026, WELL-00027, WELL-00028, WELL-00029, WELL-00031, WELL-00032, WELL-00034, WELL-00035, WELL-00036, WELL-00037, WELL-00038, WELL-00039, WELL-00040, WELL-00041, WELL-00042
Rotulo   1: WELL-00002
Rotulo   2: WELL-00002, WELL-00009, WELL-00011, WELL-00012, WELL-00013
Rotulo   3: WELL-00014
Rotulo   4: WELL-00002, WELL-00007, WELL-00014
Rotulo   5: WELL-00020
Rotulo   6: WELL-00002
Rotulo   7: WELL-00021
Rotulo   8: WELL-00025, WELL-00026, WELL-00027, WELL-00028, WELL-00029, WELL-00031, WELL-00032
Rotulo   9: WELL-00014, WELL-00042
Rotulo 101: WELL-00002
Rotulo 102: WELL-00002, WELL-00009, WELL-00011, WELL-00012, WELL-00013
Rotulo 103: Ausente
Rotulo 104: Ausente
Rotulo 105: WELL-00020
Rotulo 106: WELL-00002
Rotulo 107: WELL-00021, WELL-00022, WELL-00023
Rotulo 108: WELL-00025, WELL-00026, WEL

In [ ]:

# SESSAO 12 - TAREFA 03
# Comparacao das estrategias de tratamento dos transientes

# Trabalhar apenas com os metadados necessarios
base_rotulos = matriz_v4[
    ["Arquivo", "Poco", "Rotulo_original"]
].copy()

# CENARIO A
# Normal + eventos desenvolvidos
cenario_A = base_rotulos[
    base_rotulos["Rotulo_original"].between(0, 9)
].copy()

cenario_A["Rotulo_modelo"] = (
    cenario_A["Rotulo_original"]
)

# CENARIO B
# Associar transientes aos respectivos eventos
cenario_B = base_rotulos.copy()

cenario_B["Rotulo_modelo"] = np.where(
    cenario_B["Rotulo_original"].between(101, 109),
    cenario_B["Rotulo_original"] - 100,
    cenario_B["Rotulo_original"]
)

# CENARIO C
# Preservar os rotulos temporais distintos
cenario_C = base_rotulos.copy()

cenario_C["Rotulo_modelo"] = (
    cenario_C["Rotulo_original"]
)

# Funcao para auditar cada cenario
def resumir_cenario(base, rotulos):

    resumo = (
        base.groupby("Rotulo_modelo")
        .agg(
            Janelas=("Arquivo", "size"),
            Arquivos=("Arquivo", "nunique"),
            Pocos=("Poco", "nunique")
        )
        .reindex(rotulos, fill_value=0)
    )

    distribuicao = pd.crosstab(
        base["Rotulo_modelo"],
        base["Poco"]
    ).reindex(rotulos, fill_value=0)

    resumo["Concentracao_max_pct"] = (
        distribuicao.max(axis=1) /
        resumo["Janelas"].replace(0, np.nan)
        * 100
    ).round(2)

    return resumo

resumo_A = resumir_cenario(
    cenario_A, list(range(10))
)

resumo_B = resumir_cenario(
    cenario_B, list(range(10))
)

resumo_C = resumir_cenario(
    cenario_C,
    list(range(10)) + list(range(101, 110))
)

# Comparacao direta entre A e B
comparacao_AB = pd.DataFrame({
    "Janelas_A": resumo_A["Janelas"],
    "Arquivos_A": resumo_A["Arquivos"],
    "Pocos_A": resumo_A["Pocos"],
    "Concentracao_A_pct": resumo_A["Concentracao_max_pct"],
    "Janelas_B": resumo_B["Janelas"],
    "Arquivos_B": resumo_B["Arquivos"],
    "Pocos_B": resumo_B["Pocos"],
    "Concentracao_B_pct": resumo_B["Concentracao_max_pct"]
})

comparacao_AB["Ganho_janelas"] = (
    comparacao_AB["Janelas_B"] -
    comparacao_AB["Janelas_A"]
)

comparacao_AB["Ganho_pocos"] = (
    comparacao_AB["Pocos_B"] -
    comparacao_AB["Pocos_A"]
)

# Resumo geral dos cenarios
resultados_cenarios = []

for nome, base, resumo in [
    ("A - Sem transientes", cenario_A, resumo_A),
    ("B - Transientes associados", cenario_B, resumo_B),
    ("C - Rotulos distintos", cenario_C, resumo_C)
]:

    resultados_cenarios.append({
        "Cenario": nome,
        "Janelas": len(base),
        "Arquivos": base["Arquivo"].nunique(),
        "Pocos": base["Poco"].nunique(),
        "Classes_presentes": int(
            resumo["Janelas"].gt(0).sum()
        ),
        "Classes_ausentes": int(
            resumo["Pocos"].eq(0).sum()
        ),
        "Classes_1_poco": int(
            resumo["Pocos"].eq(1).sum()
        ),
        "Classes_2_pocos": int(
            resumo["Pocos"].eq(2).sum()
        ),
        "Classes_3_ou_mais_pocos": int(
            resumo["Pocos"].ge(3).sum()
        ),
        "Retencao_pct": round(
            len(base) / len(matriz_v4) * 100, 2
        )
    })

resumo_cenarios = pd.DataFrame(resultados_cenarios)

# Conferencias
assert len(cenario_A) == 172454
assert len(cenario_B) == 321929
assert len(cenario_C) == 321929

assert resumo_A["Janelas"].sum() == len(cenario_A)
assert resumo_B["Janelas"].sum() == len(cenario_B)
assert resumo_C["Janelas"].sum() == len(cenario_C)

# Apresentacao
print("1. COMPARACAO GERAL DOS CENARIOS:")
display(resumo_cenarios)

print("\n2. IMPACTO DA ASSOCIACAO DOS TRANSIENTES:")
display(comparacao_AB)

print("\n3. CENARIO C - ROTULOS DISTINTOS:")
display(resumo_C)

print("\n4. CLASSES COM GANHO DE COBERTURA POR POCO:")
display(
    comparacao_AB[
        comparacao_AB["Ganho_pocos"] > 0
    ]
)

# Salvamento dos resultados
resumo_cenarios.to_csv(
    pasta_s12 / "12_Comparacao_Cenarios_Transientes.csv",
    index=False
)

comparacao_AB.to_csv(
    pasta_s12 / "12_Impacto_Associacao_Transientes.csv"
)

resumo_C.to_csv(
    pasta_s12 / "12_Cenario_Rotulos_Distintos.csv"
)

print("\nRESULTADOS SALVOS EM:")
print(pasta_s12)

print("\nTAREFA 03 EXECUTADA")


1. COMPARACAO GERAL DOS CENARIOS:


,Cenario,Janelas,Arquivos,Pocos,Classes_presentes,Classes_ausentes,Classes_1_poco,Classes_2_pocos,Classes_3_ou_mais_pocos,Retencao_pct
0,A - Sem transientes,172454,478,27,10,0,5,1,4,53.57
1,B - Transientes associados,321929,478,27,10,0,4,0,6,100.00
2,C - Rotulos distintos,321929,478,27,17,2,8,1,8,100.00



2. IMPACTO DA ASSOCIACAO DOS TRANSIENTES:


,Janelas_A,Arquivos_A,Pocos_A,Concentracao_A_pct,Janelas_B,Arquivos_B,Pocos_B,Concentracao_B_pct,Ganho_janelas,Ganho_pocos
Rotulo_modelo,,,,,,,,,,
0,138404,304,27,52.94,138404,304,27,52.94,0,0
1,58,1,1,100.00,188,1,1,100.00,130,0
2,299,7,5,86.29,1700,18,5,79.82,1401,0
3,10967,31,1,100.00,10967,31,1,100.00,0,0
4,20211,143,3,78.36,20211,143,3,78.36,0,0
5,68,3,1,100.00,6105,6,1,100.00,6037,0
6,248,3,1,100.00,335,3,1,100.00,87,0
7,34,1,1,100.00,80973,14,3,65.29,80939,2
8,1856,7,7,40.68,58980,9,7,30.47,57124,0



3. CENARIO C - ROTULOS DISTINTOS:


,Janelas,Arquivos,Pocos,Concentracao_max_pct
Rotulo_modelo,,,,
0,138404,304,27,52.94
1,58,1,1,100.00
2,299,7,5,86.29
3,10967,31,1,100.00
4,20211,143,3,78.36
5,68,3,1,100.00
6,248,3,1,100.00
7,34,1,1,100.00
8,1856,7,7,40.68



4. CLASSES COM GANHO DE COBERTURA POR POCO:


,Janelas_A,Arquivos_A,Pocos_A,Concentracao_A_pct,Janelas_B,Arquivos_B,Pocos_B,Concentracao_B_pct,Ganho_janelas,Ganho_pocos
Rotulo_modelo,,,,,,,,,,
7,34,1,1,100.00,80973,14,3,65.29,80939,2
9,309,2,2,98.71,4066,13,6,46.78,3757,4



RESULTADOS SALVOS EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/12_Auditoria_Classes_V4

TAREFA 03 EXECUTADA


In [ ]:

# SESSAO 12 - TAREFA 04
# Definicao objetiva das classes elegiveis

N_FOLDS = 3
MIN_POCOS = N_FOLDS

cenarios = {
    "A": (cenario_A, resumo_A),
    "B": (cenario_B, resumo_B),
    "C": (cenario_C, resumo_C)
}

classes_elegiveis = {}
bases_elegiveis = {}
registros_decisao = []
resultados_elegibilidade = []

for nome, (base, resumo) in cenarios.items():

    tabela = resumo.copy().reset_index()

    tabela["Cenario"] = nome

    tabela["Elegivel"] = (
        tabela["Pocos"] >= MIN_POCOS
    )

    tabela["Justificativa"] = np.select(
        [
            tabela["Pocos"] == 0,
            tabela["Pocos"] == 1,
            tabela["Pocos"] == 2
        ],
        [
            "Classe ausente",
            "Insuficiente para generalizacao entre pocos",
            "Insuficiente para cobertura em tres folds"
        ],
        default="Cobertura candidata para tres folds"
    )

    elegiveis = tabela.loc[
        tabela["Elegivel"], "Rotulo_modelo"
    ].tolist()

    excluidas = tabela.loc[
        ~tabela["Elegivel"], "Rotulo_modelo"
    ].tolist()

    classes_elegiveis[nome] = elegiveis

    # Filtrar somente a base experimental derivada
    # A matriz original permanece intacta
    base_elegivel = base[
        base["Rotulo_modelo"].isin(elegiveis)
    ].copy()

    bases_elegiveis[nome] = base_elegivel

    registros_decisao.append(tabela)

    resultados_elegibilidade.append({
        "Cenario": nome,
        "Classes_elegiveis": len(elegiveis),
        "Janelas_antes": len(base),
        "Janelas_depois": len(base_elegivel),
        "Janelas_excluidas": (
            len(base) - len(base_elegivel)
        ),
        "Arquivos": base_elegivel["Arquivo"].nunique(),
        "Pocos": base_elegivel["Poco"].nunique(),
        "Retencao_cenario_pct": round(
            100 * len(base_elegivel) / len(base), 2
        ),
        "Retencao_matriz_original_pct": round(
            100 * len(base_elegivel) / len(matriz_v4), 2
        )
    })

    print(f"\nCENARIO {nome}:")
    print("Classes elegiveis:", elegiveis)
    print("Classes excluidas ou ausentes:", excluidas)

    display(
        resumir_cenario(
            base_elegivel,
            elegiveis
        )
    )

# Consolidar resultados
protocolo_classes = pd.concat(
    registros_decisao,
    ignore_index=True
)

resumo_elegibilidade = pd.DataFrame(
    resultados_elegibilidade
)

print("\n1. COMPARACAO APOS CRITERIO DE ELEGIBILIDADE:")
display(resumo_elegibilidade)

# Verificar a composicao do cenario principal
base_principal = bases_elegiveis["C"]

print("\n2. CENARIO PRINCIPAL - ROTULOS DISTINTOS:")

display(
    pd.crosstab(
        base_principal["Rotulo_modelo"],
        base_principal["Poco"]
    )
)

print("\n3. DISTRIBUICAO POR FASE:")

fases = np.select(
    [
        base_principal["Rotulo_original"] == 0,
        base_principal["Rotulo_original"] >= 101
    ],
    [
        "Normal",
        "Transiente"
    ],
    default="Evento desenvolvido"
)

distribuicao_fases = (
    pd.Series(fases)
    .value_counts()
    .rename_axis("Fase")
    .reset_index(name="Janelas")
)

distribuicao_fases["Percentual"] = (
    100 * distribuicao_fases["Janelas"] /
    len(base_principal)
).round(2)

display(distribuicao_fases)

# Conferencias
assert classes_elegiveis["A"] == [0, 2, 4, 8]
assert classes_elegiveis["B"] == [0, 2, 4, 7, 8, 9]

assert classes_elegiveis["C"] == [
    0, 2, 4, 8, 102, 107, 108, 109
]

assert len(matriz_v4) == 321929

# Salvamento
protocolo_classes.to_csv(
    pasta_s12 / "12_Protocolo_Elegibilidade_Classes.csv",
    index=False
)

resumo_elegibilidade.to_csv(
    pasta_s12 / "12_Resumo_Elegibilidade_Cenarios.csv",
    index=False
)

distribuicao_fases.to_csv(
    pasta_s12 / "12_Distribuicao_Fases_Cenario_C.csv",
    index=False
)

print("\nRESULTADOS SALVOS EM:")
print(pasta_s12)

print("\nTAREFA 04 EXECUTADA")



CENARIO A:
Classes elegiveis: [0, 2, 4, 8]
Classes excluidas ou ausentes: [1, 3, 5, 6, 7, 9]


,Janelas,Arquivos,Pocos,Concentracao_max_pct
Rotulo_modelo,,,,
0,138404,304,27,52.94
2,299,7,5,86.29
4,20211,143,3,78.36
8,1856,7,7,40.68



CENARIO B:
Classes elegiveis: [0, 2, 4, 7, 8, 9]
Classes excluidas ou ausentes: [1, 3, 5, 6]


,Janelas,Arquivos,Pocos,Concentracao_max_pct
Rotulo_modelo,,,,
0,138404,304,27,52.94
2,1700,18,5,79.82
4,20211,143,3,78.36
7,80973,14,3,65.29
8,58980,9,7,30.47
9,4066,13,6,46.78



CENARIO C:
Classes elegiveis: [0, 2, 4, 8, 102, 107, 108, 109]
Classes excluidas ou ausentes: [1, 3, 5, 6, 7, 9, 101, 103, 104, 105, 106]


,Janelas,Arquivos,Pocos,Concentracao_max_pct
Rotulo_modelo,,,,
0,138404,304,27,52.94
2,299,7,5,86.29
4,20211,143,3,78.36
8,1856,7,7,40.68
102,1401,18,5,78.44
107,80939,14,3,65.32
108,57124,9,7,31.26
109,3757,13,6,42.51


... (3 outputs left out)

Poco,WELL-00002,WELL-00007,WELL-00009,WELL-00011,WELL-00012,WELL-00013,WELL-00014,WELL-00020,WELL-00021,WELL-00022,...,WELL-00032,WELL-00034,WELL-00035,WELL-00036,WELL-00037,WELL-00038,WELL-00039,WELL-00040,WELL-00041,WELL-00042
Rotulo_modelo,,,,,,,,,,,,,,,,,,,,,
0,73274,284,69,808,217,71,991,1072,883,8588,...,722,3717,5194,350,1306,2407,1178,355,19330,983
2,2,0,20,258,6,13,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,15838,1417,0,0,0,0,2956,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,171,0,0,0,0,0,0,0,0,0
102,101,0,40,1099,46,115,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
107,0,0,0,0,0,0,0,0,830,52868,...,0,0,0,0,0,0,0,0,0,0
108,0,0,0,0,0,0,0,0,0,0,...,1714,0,0,0,0,0,0,0,0,0
109,0,0,0,0,0,0,1375,2,0,0,...,0,0,0,0,292,0,0,258,233,1597



3. DISTRIBUICAO POR FASE:


,Fase,Janelas,Percentual
0,Transiente,143221,47.11
1,Normal,138404,45.53
2,Evento desenvolvido,22366,7.36



RESULTADOS SALVOS EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/12_Auditoria_Classes_V4

TAREFA 04 EXECUTADA


In [ ]:

# SESSAO 12 - TAREFA 05
# Construcao dos folds independentes por poco

from sklearn.model_selection import PredefinedSplit

SEED = 42
N_CANDIDATOS = 30000

rng = np.random.default_rng(SEED)

# Utilizar o cenario principal como referencia
pocos = sorted(base_principal["Poco"].unique())
classes = classes_elegiveis["C"]

assert len(pocos) == 27
assert len(pocos) % N_FOLDS == 0

pocos_por_fold = len(pocos) // N_FOLDS

# Matriz de contagem: pocos x classes
contagens = pd.crosstab(
    base_principal["Poco"],
    base_principal["Rotulo_modelo"]
).reindex(
    index=pocos,
    columns=classes,
    fill_value=0
)

valores = contagens.to_numpy()
totais_classes = valores.sum(axis=0)

# Buscar distribuicoes candidatas
melhor_score = np.inf
melhor_divisao = None
n_viaveis = 0

for _ in range(N_CANDIDATOS):

    divisao = rng.permutation(
        len(pocos)
    ).reshape(N_FOLDS, pocos_por_fold)

    contagem_folds = valores[divisao].sum(axis=1)

    # Todas as classes devem aparecer em cada fold
    if (contagem_folds == 0).any():
        continue

    n_viaveis += 1

    # Equilibrio relativo das classes
    proporcoes = (
        contagem_folds / totais_classes
    )

    desequilibrio_classes = np.mean(
        (proporcoes - 1 / N_FOLDS) ** 2
    )

    # Equilibrio da quantidade total de janelas
    proporcoes_totais = (
        contagem_folds.sum(axis=1) /
        valores.sum()
    )

    desequilibrio_total = np.mean(
        (proporcoes_totais - 1 / N_FOLDS) ** 2
    )

    # Criterio baseado exclusivamente nas contagens
    score = (
        desequilibrio_classes +
        desequilibrio_total
    )

    if score < melhor_score:

        melhor_score = score
        melhor_divisao = divisao.copy()

assert melhor_divisao is not None, (
    "Nenhuma divisao viavel encontrada."
)

# Registrar a atribuicao fixa de cada poco
registros_folds = []

for fold, indices in enumerate(
    melhor_divisao, start=1
):

    for indice in indices:

        registros_folds.append({
            "Poco": pocos[indice],
            "Fold": fold
        })

mapa_folds = pd.DataFrame(
    registros_folds
).sort_values(
    ["Fold", "Poco"]
).reset_index(drop=True)

assert mapa_folds["Poco"].is_unique
assert len(mapa_folds) == 27

assert (
    mapa_folds["Fold"].value_counts() == 9
).all()

fold_por_poco = mapa_folds.set_index(
    "Poco"
)["Fold"]

print("1. BUSCA DOS FOLDS:")
print("Semente:", SEED)
print("Divisoes candidatas:", N_CANDIDATOS)
print("Divisoes viaveis:", n_viaveis)
print("Criterio de desequilibrio:", melhor_score)

print("\n2. POCOS POR FOLD:")

for fold in range(1, N_FOLDS + 1):

    lista = mapa_folds.loc[
        mapa_folds["Fold"] == fold,
        "Poco"
    ].tolist()

    print(f"\nFold {fold}:")
    print(", ".join(lista))

# Verificar os mesmos folds nos tres cenarios
resumos_folds = []
coberturas_folds = []

for nome, base in bases_elegiveis.items():

    base["Fold"] = base["Poco"].map(
        fold_por_poco
    )

    assert base["Fold"].notna().all()

    cobertura = pd.crosstab(
        base["Rotulo_modelo"],
        base["Fold"]
    ).reindex(
        index=classes_elegiveis[nome],
        columns=range(1, N_FOLDS + 1),
        fill_value=0
    )

    # Todas as classes presentes nos tres folds
    assert (cobertura > 0).all().all(), (
        f"Cobertura insuficiente no cenario {nome}"
    )

    print(f"\n3. COBERTURA - CENARIO {nome}:")
    display(cobertura)

    cobertura_long = (
        cobertura.stack()
        .reset_index(name="Janelas")
    )

    cobertura_long["Cenario"] = nome

    coberturas_folds.append(cobertura_long)

    for fold in range(1, N_FOLDS + 1):

        tr = base[base["Fold"] != fold]
        va = base[base["Fold"] == fold]

        # Garantir independencia dos pocos e arquivos
        assert set(tr["Poco"]).isdisjoint(
            va["Poco"]
        )

        assert set(tr["Arquivo"]).isdisjoint(
            va["Arquivo"]
        )

        assert tr["Poco"].nunique() == 18
        assert va["Poco"].nunique() == 9

        assert set(tr["Rotulo_modelo"]) == set(
            classes_elegiveis[nome]
        )

        assert set(va["Rotulo_modelo"]) == set(
            classes_elegiveis[nome]
        )

        resumos_folds.append({
            "Cenario": nome,
            "Fold": fold,
            "Pocos_treino": tr["Poco"].nunique(),
            "Pocos_validacao": va["Poco"].nunique(),
            "Janelas_treino": len(tr),
            "Janelas_validacao": len(va),
            "Classes_treino": tr[
                "Rotulo_modelo"
            ].nunique(),
            "Classes_validacao": va[
                "Rotulo_modelo"
            ].nunique()
        })

resumo_folds = pd.DataFrame(resumos_folds)

cobertura_final = pd.concat(
    coberturas_folds,
    ignore_index=True
)

print("\n4. RESUMO DA VALIDACAO CRUZADA:")
display(resumo_folds)

# Preparar os folds para uso futuro no scikit-learn
cv_principal = PredefinedSplit(
    test_fold=base_principal["Fold"].to_numpy(
        dtype=int
    ) - 1
)

assert cv_principal.get_n_splits() == 3

# Salvar o protocolo de validacao
mapa_folds.to_csv(
    pasta_s12 / "12_Mapa_Folds_por_Poco.csv",
    index=False
)

resumo_folds.to_csv(
    pasta_s12 / "12_Resumo_Folds_Cenarios.csv",
    index=False
)

cobertura_final.to_csv(
    pasta_s12 / "12_Cobertura_Classes_Folds.csv",
    index=False
)

print("\nRESULTADOS SALVOS EM:")
print(pasta_s12)

print("\nTAREFA 05 EXECUTADA")


1. BUSCA DOS FOLDS:
Semente: 42
Divisoes candidatas: 30000
Divisoes viaveis: 984
Criterio de desequilibrio: 0.06529776462546492

2. POCOS POR FOLD:

Fold 1:
WELL-00002, WELL-00009, WELL-00012, WELL-00020, WELL-00021, WELL-00027, WELL-00028, WELL-00036, WELL-00040

Fold 2:
WELL-00013, WELL-00014, WELL-00023, WELL-00026, WELL-00029, WELL-00031, WELL-00032, WELL-00035, WELL-00039

Fold 3:
WELL-00007, WELL-00011, WELL-00022, WELL-00025, WELL-00034, WELL-00037, WELL-00038, WELL-00041, WELL-00042

3. COBERTURA - CENARIO A:


Fold,1,2,3
Rotulo_modelo,,,
0,78072,21195,39137
2,28,13,258
4,15838,2956,1417
8,685,416,755



3. COBERTURA - CENARIO B:


Fold,1,2,3
Rotulo_modelo,,,
0,78072,21195,39137
2,215,128,1357
4,15838,2956,1417
7,864,27241,52868
8,18095,28233,12652
9,260,1379,2427



3. COBERTURA - CENARIO C:


Fold,1,2,3
Rotulo_modelo,,,
0,78072,21195,39137
2,28,13,258
4,15838,2956,1417
8,685,416,755
102,187,115,1099
107,830,27241,52868
108,17410,27817,11897
109,260,1375,2122



4. RESUMO DA VALIDACAO CRUZADA:


,Cenario,Fold,Pocos_treino,Pocos_validacao,Janelas_treino,Janelas_validacao,Classes_treino,Classes_validacao
0,A,1,18,9,66147,94623,4,4
1,A,2,18,9,136190,24580,4,4
2,A,3,18,9,119203,41567,4,4
3,B,1,18,9,190990,113344,6,6
4,B,2,18,9,223202,81132,6,6
5,B,3,18,9,194476,109858,6,6
6,C,1,18,9,190681,113310,8,8
7,C,2,18,9,222863,81128,8,8
8,C,3,18,9,194438,109553,8,8



RESULTADOS SALVOS EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/12_Auditoria_Classes_V4

TAREFA 05 EXECUTADA


In [ ]:

# SESSAO 12 - TAREFA 06
# Auditoria dos atributos ausentes e estrategia de imputacao

# 1. Auditoria geral dos atributos

X_original = matriz_v4[preditores]

ausentes = X_original.isna()
infinitos = np.isinf(X_original)

auditoria_atributos = pd.DataFrame({
    "Atributo": preditores,
    "Tipo": [
        "Estatistico" if a in atributos
        else "Indicador"
        for a in preditores
    ],
    "Valores_ausentes": ausentes.sum().values,
    "Valores_infinitos": infinitos.sum().values,
    "Valores_finitos": np.isfinite(
        X_original
    ).sum().values
})

auditoria_atributos["Ausentes_pct"] = (
    auditoria_atributos["Valores_ausentes"] /
    len(matriz_v4) * 100
).round(2)

print("1. AUDITORIA DOS 30 PREDITORES:")
display(auditoria_atributos)

# 2. Disponibilidade dos sensores

registros_sensores = []

for sensor in SENSORES:

    indicador = f"{sensor}_disponivel"

    colunas_sensor = [
        f"{sensor}_{a}"
        for a in ESTATISTICAS
    ]

    assert matriz_v4[indicador].isin([0, 1]).all()

    disponivel = matriz_v4[indicador].eq(1)
    indisponivel = matriz_v4[indicador].eq(0)

    valores_sensor = matriz_v4[colunas_sensor]

    registros_sensores.append({
        "Sensor": sensor,
        "Janelas_disponiveis": int(disponivel.sum()),
        "Janelas_indisponiveis": int(indisponivel.sum()),
        "Disponibilidade_pct": round(
            disponivel.mean() * 100, 2
        ),
        "Disponivel_com_atributo_nao_finito": int(
            (
                disponivel &
                ~np.isfinite(valores_sensor).all(axis=1)
            ).sum()
        ),
        "Indisponivel_com_atributo_preenchido": int(
            (
                indisponivel &
                valores_sensor.notna().any(axis=1)
            ).sum()
        )
    })

auditoria_sensores = pd.DataFrame(
    registros_sensores
)

assert (
    matriz_v4[indicadores].sum(axis=1) ==
    matriz_v4["N_sensores"]
).all()

print("\n2. DISPONIBILIDADE E CONSISTENCIA DOS SENSORES:")
display(auditoria_sensores)

print("\n3. QUANTIDADE DE SENSORES POR JANELA:")

distribuicao_sensores = (
    matriz_v4["N_sensores"]
    .value_counts()
    .sort_index()
    .rename_axis("N_sensores")
    .reset_index(name="Janelas")
)

distribuicao_sensores["Percentual"] = (
    distribuicao_sensores["Janelas"] /
    len(matriz_v4) * 100
).round(2)

display(distribuicao_sensores)

# 3. Disponibilidade dos sensores por classe
# Cenario principal

assert matriz_v4.index.is_unique

dados_principal = matriz_v4.loc[
    base_principal.index
].copy()

assert np.array_equal(
    dados_principal["Rotulo_original"].to_numpy(),
    base_principal["Rotulo_modelo"].to_numpy()
)

dados_principal["Rotulo_modelo"] = (
    base_principal["Rotulo_modelo"]
)

disponibilidade_por_classe = (
    dados_principal.groupby("Rotulo_modelo")[indicadores]
    .mean()
    .mul(100)
    .round(2)
)

print("\n4. DISPONIBILIDADE DOS SENSORES POR CLASSE (%):")
display(disponibilidade_por_classe)

# 4. Auditoria dos atributos dentro dos folds
# Nenhuma imputacao sera aplicada nesta etapa

registros_folds_atributos = []

for nome, base in bases_elegiveis.items():

    for fold in range(1, N_FOLDS + 1):

        idx_tr = base.index[base["Fold"] != fold]
        idx_va = base.index[base["Fold"] == fold]

        X_tr = matriz_v4.loc[idx_tr, atributos]
        X_va = matriz_v4.loc[idx_va, atributos]

        finitos_tr = np.isfinite(X_tr).sum()
        finitos_va = np.isfinite(X_va).sum()

        for atributo in atributos:

            registros_folds_atributos.append({
                "Cenario": nome,
                "Fold": fold,
                "Atributo": atributo,
                "Finitos_treino": int(
                    finitos_tr[atributo]
                ),
                "Finitos_validacao": int(
                    finitos_va[atributo]
                ),
                "Nao_finitos_treino_pct": round(
                    100 * (
                        1 - finitos_tr[atributo] / len(X_tr)
                    ), 2
                ),
                "Nao_finitos_validacao_pct": round(
                    100 * (
                        1 - finitos_va[atributo] / len(X_va)
                    ), 2
                ),
                "Sem_dados_no_treino": bool(
                    finitos_tr[atributo] == 0
                )
            })

auditoria_atributos_folds = pd.DataFrame(
    registros_folds_atributos
)

resumo_imputacao = (
    auditoria_atributos_folds
    .groupby(["Cenario", "Fold"])
    .agg(
        Atributos_auditados=("Atributo", "size"),
        Atributos_sem_dados_treino=(
            "Sem_dados_no_treino", "sum"
        ),
        Maior_ausencia_treino_pct=(
            "Nao_finitos_treino_pct", "max"
        ),
        Maior_ausencia_validacao_pct=(
            "Nao_finitos_validacao_pct", "max"
        )
    )
    .reset_index()
)

print("\n5. VIABILIDADE DA IMPUTACAO POR FOLD:")
display(resumo_imputacao)

print("\n6. ATRIBUTOS SEM DADOS NO TREINAMENTO:")

problemas_imputacao = auditoria_atributos_folds[
    auditoria_atributos_folds["Sem_dados_no_treino"]
]

if problemas_imputacao.empty:
    print("Nenhum atributo completamente ausente no treino.")
else:
    display(problemas_imputacao)

# 5. Salvamento dos resultados

auditoria_atributos.to_csv(
    pasta_s12 / "12_Auditoria_Atributos_Ausentes.csv",
    index=False
)

auditoria_sensores.to_csv(
    pasta_s12 / "12_Auditoria_Disponibilidade_Sensores.csv",
    index=False
)

distribuicao_sensores.to_csv(
    pasta_s12 / "12_Distribuicao_Numero_Sensores.csv",
    index=False
)

disponibilidade_por_classe.to_csv(
    pasta_s12 / "12_Disponibilidade_Sensores_por_Classe.csv"
)

auditoria_atributos_folds.to_csv(
    pasta_s12 / "12_Auditoria_Atributos_por_Fold.csv",
    index=False
)

resumo_imputacao.to_csv(
    pasta_s12 / "12_Viabilidade_Imputacao.csv",
    index=False
)

assert matriz_v4.shape == (321929, 37)

print("\nRESULTADOS SALVOS EM:")
print(pasta_s12)

print("\nTAREFA 06 EXECUTADA")


1. AUDITORIA DOS 30 PREDITORES:


,Atributo,Tipo,Valores_ausentes,Valores_infinitos,Valores_finitos,Ausentes_pct
0,P-TPT_media,Estatistico,56612,0,265317,17.59
1,P-TPT_desvio,Estatistico,56612,0,265317,17.59
2,P-TPT_inclinacao,Estatistico,56612,0,265317,17.59
3,P-TPT_p10,Estatistico,56612,0,265317,17.59
4,P-TPT_p90,Estatistico,56612,0,265317,17.59
5,T-TPT_media,Estatistico,59674,0,262255,18.54
6,T-TPT_desvio,Estatistico,59674,0,262255,18.54
7,T-TPT_inclinacao,Estatistico,59674,0,262255,18.54
8,T-TPT_p10,Estatistico,59674,0,262255,18.54
9,T-TPT_p90,Estatistico,59674,0,262255,18.54



2. DISPONIBILIDADE E CONSISTENCIA DOS SENSORES:


,Sensor,Janelas_disponiveis,Janelas_indisponiveis,Disponibilidade_pct,Disponivel_com_atributo_nao_finito,Indisponivel_com_atributo_preenchido
0,P-TPT,265317,56612,82.41,0,0
1,T-TPT,262255,59674,81.46,0,0
2,P-ANULAR,252724,69205,78.50,0,0
3,P-JUS-CKGL,215165,106764,66.84,0,0
4,P-MON-CKP,229795,92134,71.38,0,0



3. QUANTIDADE DE SENSORES POR JANELA:


,N_sensores,Janelas,Percentual
0,2,11146,3.46
1,3,154023,47.84
2,4,42905,13.33
3,5,113855,35.37



4. DISPONIBILIDADE DOS SENSORES POR CLASSE (%):


,P-TPT_disponivel,T-TPT_disponivel,P-ANULAR_disponivel,P-JUS-CKGL_disponivel,P-MON-CKP_disponivel
Rotulo_modelo,,,,,
0,91.77,86.28,86.16,45.03,46.42
2,100.00,100.00,100.00,6.69,8.70
4,100.00,100.00,92.98,21.62,21.61
8,93.97,85.08,56.95,59.32,100.00
102,100.00,100.00,100.00,2.86,6.14
107,66.34,87.03,66.34,99.99,100.00
108,68.74,47.67,64.48,79.17,100.00
109,100.00,100.00,92.23,100.00,100.00



5. VIABILIDADE DA IMPUTACAO POR FOLD:


,Cenario,Fold,Atributos_auditados,Atributos_sem_dados_treino,Maior_ausencia_treino_pct,Maior_ausencia_validacao_pct
0,A,1,25,0,29.43,94.41
1,A,2,25,0,68.19,57.84
2,A,3,25,0,75.02,21.98
3,B,1,25,0,29.64,78.95
4,B,2,25,0,47.50,69.76
5,B,3,25,0,46.12,15.05
6,C,1,25,0,29.68,78.97
7,C,2,25,0,47.57,69.76
8,C,3,25,0,46.13,15.09



6. ATRIBUTOS SEM DADOS NO TREINAMENTO:
Nenhum atributo completamente ausente no treino.

RESULTADOS SALVOS EM:
/content/drive/MyDrive/Tópicos Especiais IA/EXPERIMENTOS_EXTRAS_C1/EXP02_Preprocessamento_V4/12_Auditoria_Classes_V4

TAREFA 06 EXECUTADA


In [ ]:

# SESSAO 12 - TAREFA 07
# Consolidacao do protocolo experimental V4

import json
import hashlib

# 1. Conferencia final da matriz original

assert matriz_v4.shape == (321929, 37)
assert matriz_v4.index.is_unique
assert len(preditores) == 30
assert len(atributos) == 25
assert len(indicadores) == 5

assert auditoria_atributos["Valores_infinitos"].sum() == 0

assert (
    auditoria_sensores[
        "Disponivel_com_atributo_nao_finito"
    ].sum() == 0
)

assert (
    auditoria_sensores[
        "Indisponivel_com_atributo_preenchido"
    ].sum() == 0
)

assert problemas_imputacao.empty

# 2. Consolidacao dos cenarios

descricoes = {
    "A": "Normal e eventos desenvolvidos, sem transientes",
    "B": "Transientes associados aos respectivos eventos",
    "C": "Transientes e eventos desenvolvidos distintos"
}

janelas_esperadas = {
    "A": 160770,
    "B": 304334,
    "C": 303991
}

config_cenarios = {}

for nome, base in bases_elegiveis.items():

    elegiveis = [
        int(r) for r in classes_elegiveis[nome]
    ]

    assert len(base) == janelas_esperadas[nome]
    assert base.index.is_unique
    assert base["Fold"].notna().all()

    assert set(base["Rotulo_modelo"]) == set(
        elegiveis
    )

    assert base["Fold"].eq(
        base["Poco"].map(fold_por_poco)
    ).all()

    originais = matriz_v4.loc[
        base.index, "Rotulo_original"
    ].to_numpy()

    if nome == "B":

        rotulos_esperados = np.where(
            (originais >= 101) &
            (originais <= 109),
            originais - 100,
            originais
        )

    else:
        rotulos_esperados = originais

    assert np.array_equal(
        base["Rotulo_modelo"].to_numpy(),
        rotulos_esperados
    )

    rotulos_possiveis_cenario = (
        list(range(10)) +
        list(range(101, 110))
        if nome == "C"
        else list(range(10))
    )

    config_cenarios[nome] = {
        "descricao": descricoes[nome],
        "classes_elegiveis": elegiveis,
        "classes_excluidas_ou_ausentes": [
            r for r in rotulos_possiveis_cenario
            if r not in elegiveis
        ],
        "janelas": int(len(base)),
        "arquivos": int(
            base["Arquivo"].nunique()
        ),
        "pocos": int(
            base["Poco"].nunique()
        )
    }

# 3. Consolidacao dos folds

config_folds = {}

for fold in range(1, N_FOLDS + 1):

    pocos_validacao = sorted(
        mapa_folds.loc[
            mapa_folds["Fold"] == fold,
            "Poco"
        ].tolist()
    )

    pocos_treinamento = sorted(
        mapa_folds.loc[
            mapa_folds["Fold"] != fold,
            "Poco"
        ].tolist()
    )

    assert len(pocos_validacao) == 9
    assert len(pocos_treinamento) == 18

    assert set(pocos_validacao).isdisjoint(
        pocos_treinamento
    )

    config_folds[str(fold)] = {
        "pocos_treinamento": pocos_treinamento,
        "pocos_validacao": pocos_validacao
    }

# 4. Configuracao experimental completa

protocolo_s12 = {
    "projeto": "C1 - Classificacao supervisionada 3W",
    "dataset": "3W 2.0.0",
    "sessao": 12,
    "data": "2026-11-25",
    "particionamento": "V4 ajustada",
    "origem_dados": "Real",
    "cenario_principal": "C",
    "cenarios_comparativos": ["A", "B"],

    "matriz_original": {
        "janelas": 321929,
        "colunas": 37,
        "arquivos": 478,
        "pocos": 27,
        "janela_amostras": 100,
        "deslocamento_amostras": 50,
        "minimo_sensores": 2,
        "rotulos_homogeneos": True
    },

    "preditores": {
        "sensores": SENSORES,
        "estatisticas": ESTATISTICAS,
        "atributos_estatisticos": atributos,
        "indicadores_disponibilidade": indicadores,
        "ordem_preditores": preditores
    },

    "elegibilidade": {
        "minimo_pocos_por_classe": MIN_POCOS,
        "criterio": (
            "Representatividade em tres ou mais "
            "pocos do treinamento"
        ),
        "escopo": "Classificacao de classes elegiveis",
        "preservar_matriz_original": True,
        "cenarios": config_cenarios
    },

    "validacao_cruzada": {
        "metodo": (
            "Folds fixos por poco com busca "
            "baseada nas contagens de classes"
        ),
        "n_folds": N_FOLDS,
        "semente": SEED,
        "divisoes_candidatas": N_CANDIDATOS,
        "divisoes_viaveis": int(n_viaveis),
        "criterio_desequilibrio": float(
            melhor_score
        ),
        "otimizacao_por_metricas_modelo": False,
        "folds": config_folds
    },

    "preprocessamento": {
        "imputacao": "Mediana",
        "ajuste_imputacao": (
            "Exclusivamente no treinamento de cada fold"
        ),
        "preservar_indicadores_binarios": True,
        "normalizacao": (
            "Quando necessaria, ajustada no treino"
        ),
        "metadados_como_preditores": False,
        "balanceamento": (
            "Somente no treinamento; "
            "estrategia a avaliar na Sessao 13"
        )
    },

    "experimentos_planejados": {
        "baseline_sem_balanceamento": True,
        "avaliar_ponderacao_classes": True,
        "ablacao_disponibilidade": [
            "30 preditores",
            "25 atributos estatisticos",
            "5 indicadores de disponibilidade"
        ],
        "sensibilidade_dados_sinteticos": True
    },

    "controle_experimental": {
        "validacao_fixa_V4_preservada": True,
        "teste_fixo_V4_preservado": True,
        "validacao_e_teste_processados": False,
        "auditar_cobertura_temporal_externa": True,
        "reportar_classes_fora_do_escopo": True,
        "modelos_treinados_nesta_sessao": False,
        "imputacao_aplicada_nesta_sessao": False
    }
}

# 5. Registrar integridade dos CSVs da sessao

arquivos_csv = sorted(
    pasta_s12.glob("12_*.csv")
)

protocolo_s12["arquivos_auditoria"] = [
    {
        "arquivo": caminho.name,
        "bytes": caminho.stat().st_size,
        "sha256": hashlib.sha256(
            caminho.read_bytes()
        ).hexdigest()
    }
    for caminho in arquivos_csv
]

# 6. Salvar o protocolo

caminho_protocolo = (
    pasta_s12 / "12_Protocolo_Experimental_V4.json"
)

with open(
    caminho_protocolo,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        protocolo_s12,
        f,
        indent=4,
        ensure_ascii=False
    )

# 7. Verificar a recuperacao do protocolo

with open(
    caminho_protocolo,
    "r",
    encoding="utf-8"
) as f:

    protocolo_recuperado = json.load(f)

assert protocolo_recuperado == protocolo_s12

# 8. Apresentacao final

print("1. PROTOCOLO EXPERIMENTAL CONSOLIDADO:")

print("Cenario principal:", "C")
print("Cenarios comparativos:", "A e B")
print("Minimo de pocos por classe:", MIN_POCOS)
print("Folds:", N_FOLDS)
print("Preditores:", len(preditores))
print("Imputacao:", "Mediana por fold")

print("\n2. CENARIOS EXPERIMENTAIS:")

for nome, config in config_cenarios.items():

    print(f"\nCenario {nome}:")
    print("Classes:", config["classes_elegiveis"])
    print("Janelas:", config["janelas"])
    print("Arquivos:", config["arquivos"])
    print("Pocos:", config["pocos"])

print("\n3. ARQUIVOS PRODUZIDOS:")

for caminho in sorted(pasta_s12.iterdir()):
    if caminho.is_file():
        print(caminho.name)

print("\n4. TOTAL DE CSVs:", len(arquivos_csv))

print("\n5. PROTOCOLO SALVO EM:")
print(caminho_protocolo)

print("\n6. CONTROLE EXPERIMENTAL:")
print("Matriz original preservada: SIM")
print("Folds definidos: SIM")
print("Imputacao aplicada: NAO")
print("Modelos treinados: NAO")
print("Teste utilizado: NAO")

print("\nTAREFA 07 EXECUTADA")


1. PROTOCOLO EXPERIMENTAL CONSOLIDADO:
Cenario principal: C
Cenarios comparativos: A e B
Minimo de pocos por classe: 3
Folds: 3
Preditores: 30
Imputacao: Mediana por fold

2. CENARIOS EXPERIMENTAIS:

Cenario A:
Classes: [0, 2, 4, 8]
Janelas: 160770
Arquivos: 447
Pocos: 27

Cenario B:
Classes: [0, 2, 4, 7, 8, 9]
Janelas: 304334
Arquivos: 447
Pocos: 27

Cenario C:
Classes: [0, 2, 4, 8, 102, 107, 108, 109]
Janelas: 303991
Arquivos: 447
Pocos: 27

3. ARQUIVOS PRODUZIDOS:
12_Auditoria_Atributos_Ausentes.csv
12_Auditoria_Atributos_por_Fold.csv
12_Auditoria_Disponibilidade_Sensores.csv
12_Auditoria_Representatividade_Classes.csv
12_Cenario_Rotulos_Distintos.csv
12_Cobertura_Classes_Folds.csv
12_Comparacao_Cenarios_Transientes.csv
12_Disponibilidade_Sensores_por_Classe.csv
12_Distribuicao_Fases_Cenario_C.csv
12_Distribuicao_Numero_Sensores.csv
12_Distribuicao_Rotulos_por_Poco.csv
12_Impacto_Associacao_Transientes.csv
12_Mapa_Folds_por_Poco.csv
12_Protocolo_Elegibilidade_Classes.csv
12_Protocol